# 12-зертханалық жұмыс. Сенімділік аралықтары: есептеу, қамту және bootstrap

**Пән:** Деректерді интеллектуалды талдау  
**Ұзақтығы:** 2 академиялық сағат  
**Алдын ала дайындық:** 11-12-дәрістер; SciPy функциялары, таңдамалық статистика және NumPy.

## Оқу мақсаттары

- z- немесе t-аралығын таңдауды негіздеу
- Сенімділік деңгейі мен таңдама көлемінің әсерін зерттеу
- Bootstrap орындап, түсіндіру аясын дұрыс шектеу

## Ноутбукты іске қосу

Python ортасында мына пакеттерді орнатыңыз: `numpy`, `pandas`, `scipy`. Jupyter бағдарламасында Python 3 ядросын таңдап, ұяшықтарды жоғарыдан төмен орындаңыз. Соңғы код ұяшығында тапсырмаларды өз бетіңізше орындаңыз. Нәтижелерді зертханалық жұмыста көрсетілген `work_results` жолына сақтаңыз.

Оқулықтың архивін ашып, `notebooks/` және `data/` қалталарын бірге сақтаңыз. Ноутбукты бөлек жүктесеңіз, Jupyter немесе Colab файлдар панелінде ағымдағы жұмыс қалтасында `data` ішкі қалтасын жасап, оқулықтан алынған `football.csv` файлын сонда орналастырыңыз. Бірінші ұяшық дерек жолын тексереді.


In [ ]:
from pathlib import Path
import os

# Жұмыс қалтасын анықтау.
project_dir = Path.cwd()
if project_dir.name == "notebooks" and (
    (project_dir.parent / "data" / "football.csv").is_file()
    or (project_dir.parent / "index.html").is_file()
):
    project_dir = project_dir.parent

if not (project_dir / "data" / "football.csv").is_file():
    raise FileNotFoundError(
        "data/football.csv табылмады. Оқулықтан football.csv файлын жүктеп алып, "
        "ағымдағы жұмыс қалтасындағы data ішкі қалтасына орналастырыңыз."
    )

if Path.cwd() != project_dir:
    os.chdir(project_dir)

print("Жұмыс қалтасы:", project_dir)


## Есеп және жұмысты ұйымдастыру

Сіз бірнеше оқу өлшеу процесі үшін белгісіздікті есептеп, кейін тарихи деректерді қайта іріктеуді көрсетесіз. Әр жағдайда параметр не екенін, қандай мәліметтер белгілі деп саналатынын, бақылаулар қалай алынғанын және аралық қандай нысанға қатысты екенін анықтау керек. Әдісті негіздемей тек шекараларды беру жеткіліксіз.

Бастапқы зертханалық жұмыста батареялар, сатып алушы шығындары, салықтар, пікірлер және критикалық мәндер туралы есептер бар. Бұл бағыттар сақталды. Таңдамасы тек суретпен берілген шарт толық берілген оқу массивімен ауыстырылды. Тек формулаға мән қоюды емес, сенімділік деңгейінің мағынасын тексеру үшін қамту симуляциясы мен bootstrap қосылды.

Python, NumPy, pandas, SciPy және matplotlib қажет. Барлық дерек жергілікті не мәтінде берілген; жүктеу қажет емес. Notebook не скриптті жинақ түбірінен іске қосып, нәтижелерді work_results ішіне жазыңыз. Симуляцияда np.random.default_rng(42) жасаңыз. Қорытынды есепте шекаралар есептелгеннен кейін тұрақты параметрдің ықтималдығы туралы тұжырым қолданбай, сенімділіктің жиіліктік түсіндірмесін пайдаланыңыз.

## Талданған мысал және бастапқы деректер

Жеке оқу есебінде sigma=15 белгілі, қалыпты процестен тәуелсіз кездейсоқ n=25 таңдама алынған, орташа мәні 100. Стандартты қате 3-ке тең. 95% аралық үшін критикалық z шамамен 1.96, сондықтан шекаралар шамамен 94.12 және 105.88. Аралық жеке өлшемге емес, процестің орташа мәніне қатысты.



In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm, t

out = Path('work_results')
out.mkdir(exist_ok=True)
sample_mean, sigma, n = 100., 15., 25
critical = norm.ppf(0.975)
half = critical * sigma / np.sqrt(n)
print(round(sample_mean - half, 2))
print(round(sample_mean + half, 2))



Белгіленген іріктеу және аралық құру рәсімін қайталасақ, модель орындалғанда аралықтардың шамамен 95%-ы шынайы орташа мәнді қамтиды. Осы есептелген аралық енді бекітілген. Бұл анықтаманы «өлшемдердің 95%-ы шекаралар арасында жатыр» тіркесімен алмастыруға болмайды. Жеке мәндер ауқымы үшін басқа талдау нысаны қажет.

3-тапсырма үшін операцияны орындау уақытының секундпен берілген толық мәндерін қолданыңыз:



In [ ]:
times = np.array([
    9.8, 10.1, 10.4, 9.9, 10.3,
    10.0, 9.7, 10.2, 10.5, 9.6,
    10.1, 10.4, 9.9, 10.2, 10.0,
    10.3, 9.8, 10.1, 10.2, 9.7,
])
assert times.size == 20 and np.isfinite(times).all()
football = pd.read_csv(Path('data') / 'football.csv')
assert football.shape == (12897, 42)
assert not football['Age'].isna().any()



times үшін тәуелсіздік пен жуық қалыптылық — оқу жорамалдары. football үшін мұндай жорамалдар анықталмаған: бұл тарихи файл, бас жиынтық белгісіз, клубтар тәуелді жолдарды біріктіруі мүмкін. Оның орташа мәнін қолдағы жолдардың орташа мәні ретінде дәл сипаттауға болады, ал төмендегі bootstrap ойыншыларды тәуелсіз таңдау шартты моделіндегі жаттығу болып табылады.

## Тапсырмалар

1. **Белгілі стандартты ауытқу.** Зауыт шынайы sigma=2.4 ай екенін хабарлайды. 64 батареядан тұратын кездейсоқ тәуелсіз таңдаманың орташа қызмет мерзімі 12.35 ай. 90% z-аралығын есептеңіз. alpha, критикалық мән, SE, жарты ен және екі шекараны шығарыңыз. Шынайы орташа мәннің өзі белгісіз екенін көрсетіңіз; таңдамалық орташа мән есептелгеннен кейін «шынайы» болып кетпейді. Аралық есептеуде есептелген квантиль орнына дөңгелектелген 1.64 қолданбаңыз.

2. **Белгісіз стандартты ауытқу.** 15 сатып алушыдан тұратын кездейсоқ тәуелсіз таңдамада орташа шығын 2000 шартты ақша бірлігі, таңдамалық s=400. Есепте шығындар үлестірімі жуық қалыпты деп берілген. 14 еркіндік дәрежесі бар 95% t-аралығын құрыңыз. Қатар қате z-нұсқасын салыстыру үшін есептеп, оның неліктен негізгі жауап емес екенін белгілеңіз. s-тің қосымша белгісіздігі қалай әсер ететінін көрсетіңіз. Тексермей бұл үлестірімді кез келген нақты шығынға қолданбаңыз.

3. **Бастапқы мәндер бойынша есептеу.** times массиві үшін n, орташа мән, ddof=1 болатын s және SE анықтаңыз. 90%, 95% және 99% t-аралықтарын құрыңыз. Жолдарды ортақ нәтижелер кестесіне сақтаңыз. Деректер өзгермегенде ен сенімділік деңгейімен бірге өсетінін және есептеу дәлдігін ескере отырып, әр аралық орташа мәнге қатысты симметриялы екенін тексеріңіз. n>=2 және 0<confidence<1 шарттарын тексеретін, массив бойынша аралық есептейтін функция жазыңыз. Қате енгізуде ол түсінікті қате беруі, ал шақырушы код оны өңдеуі керек.

4. **Үлес және рәсім таңдау.** 189 пікірден тұратын кездейсоқ тәуелсіз таңдамада 132 пікір оң. p_hat және 90% Вальд аралығын есептеңіз. Кейін Уилсон формуласын іске асырып, шекараларды салыстырыңыз. k=2, n=10 үшін тек әдістер салыстыруын қайталаңыз. Уилсон формуласы: z=norm.ppf((1+confidence)/2), d=1+z**2/n, c=(p_hat+z**2/(2*n))/d, h=z*sqrt(p_hat*(1-p_hat)/n+z**2/(4*n**2))/d, шекаралар c-h және c+h. Вальдтың теріс шекарасын қию әдісті неліктен дұрыс етпейтінін түсіндіріңіз. Пікірлер ерікті қалдырылса, ықтимал ығысуды бөлек көрсетіңіз.

5. **Дәлдік және деректер көлемі.** Белгілі sigma=12 және қалыпты процесс кезінде 95% аралық жарты ені 2-ден аспауына қажетті n табыңыз. n >= (z*sigma/2)**2 қолданып, жоғары қарай дөңгелектеңіз. Орнына қойып, есептелген n талапты орындайтынын, ал n-1 орындамайтынын тексеріңіз. Бірдей sigma және деңгей үшін n=25, 100 және 400 жарты ендерін салыстырыңыз. Квадрат түбір ережесін түсіндіріп, n арттыру іріктеу ығысуын неге жоймайтынын көрсетіңіз.

6. **Модельденетін процестегі қамту.** mu=50, sigma=10 қалыпты моделінен әрқайсысы 30 мәннен тұратын 2000 тәуелсіз таңдама жасаңыз. Әрқайсысы үшін өзінің s мәнімен 95% t-аралығын құрыңыз. 50-ді қамтитын аралықтардың үлесін есептеңіз. Алғашқы 30 қайталаудың шекаралары мен қамту индикаторын сақтаңыз; mu=50 сызығы бар осы аралықтар суретін құрыңыз. Алынған үлес дәл 0.95 болуға міндетті емес. Қай белгілі параметр қамтуды белгілеуге мүмкіндік бергенін және әдеттегі есепте оны неліктен тексере алмайтынымызды көрсетіңіз.

7. **Жас үшін bootstrap.** football ішінен Age бағанын сандық массив ретінде алыңыз. n көлеміндегі 1000 қайтарымды таңдама жасап, орташа мәндерді сақтаңыз. 0.025 және 0.975 квантильдері бойынша процентильдік аралық құрыңыз. Оқу мақсатындағы салыстыру ретінде сол массив бойынша t-аралығын қатар есептеңіз. Жеке ойыншыларды қайта іріктеу жолдардың тәуелсіздігін жорамалдайтынын, клубтық құрылым оны бұзуы мүмкін екенін жазыңыз. Ешбір аралықты барлық қазіргі футболшының орташа жасының кепілі ретінде көрсетпеңіз. Файл оқылғаннан кейін оның орташа Age мәні белгілі екенін түсіндіріңіз.

8. **Қорытындылар аудиті.** Әр дерек түрі үшін бақылау бірлігі, n, баға, әдіс, деңгей және шектеулері бар бір ұқыпты қорытынды құрыңыз. Үш қате тіркесті тауып түзетіңіз: «шынайы орташа мән есептелген аралықта 95% ықтималдықпен жатыр»; «сатып алушылардың 95%-ы L-ден U-ға дейін жұмсайды, себебі бұл орташаның аралығы»; «аралық тар, демек іріктеу репрезентативті». Бүкіл жұмысты қайта іске қосып, нәтижелер файлдарды қолмен өңдемей, берілген деректерден жасалатынына көз жеткізіңіз.

## Нәтижелер және инварианттар

Notebook не скрипт, түсіндірмелер және work_results ішіндегі lab12_intervals.csv, lab12_coverage.csv, lab12_coverage.png, lab12_bootstrap.csv файлдарын тапсырыңыз. Аралықтардың ортақ кестесінде әр case, method және confidence үйлесіміне бір нәтиже бар. Міндетті бағандарды сақтай отырып, SE мен критикалық мән үшін қосымша бағандар қосуға болады.

| Файл | Міндетті бағандар | Күтілетін тексеру |
|---|---|---|
| lab12_intervals.csv | case, method, confidence, n, estimate, lower, upper | lower<=upper, 0<confidence<1 |
| lab12_coverage.csv | repeat, lower, upper, covered | 30 жол, covered мәндері 0/1 |
| lab12_bootstrap.csv | repeat, mean_age | 1000 жол, барлық мән ақырлы |
| lab12_coverage.png | Алғашқы 30 қайталаудың аралықтары | Шынайы mu=50 белгіленген |

Симметриялы z/t әдістеріндегі орташа мән аралықтары үшін float қатесінің рұқсатымен estimate=(lower+upper)/2 теңдігін тексеріңіз. Уилсон үшін бұл теңдік p_hat-пен әдетте орындалмайды: орталық түзетілген. Барлық үлестер мен деңгейлерді сан түрінде, мысалы 0.95 деп сақтаңыз. Аргумент атауларында alpha=0.05 пен confidence=0.95 шамаларын шатастырмаңыз.

Bootstrap кезінде қажетсіз 1000-ға 12897 матрицасын жасамаңыз: бір қайталама таңдама массиві бар цикл жадты үнемдейді. Бастапқы деректердің барлық көшірмесін емес, статистиканы сақтаңыз. Операциялар тізбегі бірдей болса, бір seed бірдей нәтиже беруі керек. Ортақ генератор қолданылғанда жаңа кездейсоқ операциялар қосылса, нәтижелер өзгеруі мүмкін; жеке тәжірибелер үшін оны саналы түрде жаңадан жасаңыз.

Қосымша тапсырма: scale=5 экспоненциалдық үлестіріміне арналған t-аралығының қамтуын n=5 және n=80 кезінде салыстырыңыз. Әр жағдайда mu=5 белгілі. Нәтижені әдістің кез келген қалыпты емес үлестірімде жұмыс істеуінің дәлелі деп жарияламаңыз: ол нақты модельденген екі шартқа қатысты.

## Қорғауға арналған сұрақтар

1. Шарттағы sigma мен таңдамадағы s мәртебесі неліктен әртүрлі?
2. 95% сенімділік анықтамасында қандай кездейсоқ нысандар қайталанады?
3. Орташаның аралығы неліктен жеке шығындар аралығы емес?
4. Ойыншылар бойынша әдеттегі bootstrap-ты қандай тәуелділіктер бұзады?
5. Іріктеу жүйелі ығысқан болса, n өскенде не сақталады?

Бағалау, барлығы 100 балл:

- 20 балл — әдісті негіздейтін z- және t-есептеулер.
- 15 балл — массивке арналған функция және сенімділік деңгейінің әсері.
- 15 балл — үлестер, Вальд пен Уилсонды салыстыру.
- 15 балл — көлемді жоспарлау және жарты енді тексеру.
- 15 балл — қамту симуляциясы және сурет.
- 15 балл — football bootstrap және дұрыс шектеулер.
- 5 балл — файлдар, таза іске қосу және қорғау.

Құжаттама:

- [SciPy: t-үлестірімі](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html).
- [SciPy: bootstrap](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html).
- [NumPy: quantile](https://numpy.org/doc/stable/reference/generated/numpy.quantile.html).

## Өзіндік жұмыс

Осы зертханалық жұмыстың тапсырмаларын төмендегі ұяшықта орындаңыз. Қажет болса, код пен түсіндірме үшін қосымша ұяшықтар қосыңыз. Есептеулерді аяқтаған соң ядроны қайта іске қосып, ұяшықтарды басынан бастап орындаңыз.

In [ ]:
# Тапсырмаларды орындау кодын осы жерге жазыңыз.
